# Day 4 - Evidence Reliability Scoring + Proposed Pipeline
## Trustworthy Evidence-Grounded Framework - FEVEROUS Validation

**Objectives:**
1. Implement the 2-term Evidence Reliability Score (ERS):
   - `relevance_i` = cosine similarity (claim embedding, evidence embedding)
   - `consistency_i` = cross-type agreement (embedding similarity across different evidence types)
2. Re-rank retrieved evidence by ERS
3. Run the **Proposed system** (ERS-weighted context + Mistral-7B) across all 450 claims
4. Save `proposed_outputs.csv`

---
**Evidence types in corpus:** `sentence`, `cell`, `table_row`
The consistency term treats any pair of different types as cross-type.

**ERS formula:**
```
ERS_i = 0.7 * relevance_i + 0.3 * consistency_i
```

> **Package versions are pinned to match the working Day 1 environment.**

In [ ]:
# Cell 1: Install — versions pinned to match Day 1 working environment
!pip install -q \
    "datasets==2.15.0" \
    "huggingface_hub==0.19.4" \
    "fsspec==2023.10.0" \
    sentence-transformers \
    faiss-cpu \
    transformers \
    accelerate \
    bitsandbytes
print('Packages ready')

In [ ]:
# Cell 2: HF Login
from huggingface_hub import login
login()

In [ ]:
# Cell 3: Drive mount and paths
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/feverous-validation'
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUT_DIR  = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)
print('Drive mounted')

In [ ]:
# Cell 4: Imports
import json
import csv
import time
import numpy as np
import torch
import faiss
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.notebook import tqdm

print('CUDA:', torch.cuda.is_available())

In [ ]:
# Cell 5: Load frozen dataset and pre-computed embeddings from Day 2
with open(os.path.join(DATA_DIR, 'claims.json'), encoding='utf-8') as f:
    claims = json.load(f)

with open(os.path.join(DATA_DIR, 'evidence_corpus.json'), encoding='utf-8') as f:
    evidence_corpus = json.load(f)

# Load pre-computed embeddings — saves ~20 min vs re-embedding
corpus_embeddings = np.load(os.path.join(DATA_DIR, 'corpus_embeddings.npy'))
claim_embeddings  = np.load(os.path.join(DATA_DIR, 'claim_embeddings.npy'))
index = faiss.read_index(os.path.join(DATA_DIR, 'faiss_index.bin'))

import pandas as pd
retrieval_df = pd.read_csv(os.path.join(OUT_DIR, 'retrieval_results.csv'))

# Evidence type summary for reference
type_counts = Counter(ev['type'] for ev in evidence_corpus)
print(f'Claims: {len(claims)}, Corpus: {len(evidence_corpus)}')
print(f'Corpus embeddings shape: {corpus_embeddings.shape}')
print(f'Evidence type breakdown: {dict(type_counts)}')

In [ ]:
# Cell 6: Evidence Reliability Score (ERS) implementation
#
# The corpus contains three evidence types: 'sentence', 'cell', 'table_row'
# The consistency term uses cross-type pairs:
#   - 'sentence'  vs  'cell' or 'table_row'
#   - 'cell'      vs  'sentence' or 'table_row'
#   - 'table_row' vs  'sentence' or 'cell'
# If all retrieved items share the same type, consistency falls back to relevance.

W_RELEVANCE   = 0.7
W_CONSISTENCY = 0.3
TOP_K_RETRIEVE = 20   # Retrieve 20, re-rank by ERS, keep top 10 for prompt
TOP_K_PROMPT   = 10


def compute_ers(claim_emb, candidate_idxs, candidate_scores, corpus_embeddings, evidence_corpus):
    """
    Compute 2-term ERS for a list of candidate evidence items.

    Returns list of dicts sorted by ERS descending:
        {corpus_idx, ers, relevance, consistency}
    """
    if not candidate_idxs:
        return []

    # Term 1: Relevance (FAISS cosine similarity), normalized to [0,1]
    relevance = np.array(candidate_scores, dtype=np.float32)
    r_min, r_max = relevance.min(), relevance.max()
    if r_max > r_min:
        relevance_norm = (relevance - r_min) / (r_max - r_min)
    else:
        relevance_norm = np.ones_like(relevance)

    # Term 2: Cross-type consistency
    # Pairwise cosine similarity matrix (embeddings already L2-normalized)
    candidate_embs = corpus_embeddings[candidate_idxs]     # shape [K, dim]
    sim_matrix = candidate_embs @ candidate_embs.T         # shape [K, K]
    types = [evidence_corpus[idx]['type'] for idx in candidate_idxs]

    consistency = np.zeros(len(candidate_idxs))
    for i, t in enumerate(types):
        # Indices of items with a DIFFERENT type (all three types are considered)
        diff_type_idxs = [j for j, tj in enumerate(types) if tj != t]
        if diff_type_idxs:
            consistency[i] = sim_matrix[i, diff_type_idxs].mean()
        else:
            # All same type: fall back to relevance score
            consistency[i] = relevance_norm[i]

    # Normalize consistency to [0, 1]
    c_min, c_max = consistency.min(), consistency.max()
    if c_max > c_min:
        consistency_norm = (consistency - c_min) / (c_max - c_min)
    else:
        consistency_norm = np.ones_like(consistency)

    # Final ERS
    ers = W_RELEVANCE * relevance_norm + W_CONSISTENCY * consistency_norm

    order = np.argsort(-ers)
    return [
        {
            'corpus_idx':  candidate_idxs[o],
            'ers':         round(float(ers[o]), 5),
            'relevance':   round(float(relevance_norm[o]), 5),
            'consistency': round(float(consistency_norm[o]), 5)
        }
        for o in order
    ]


print('ERS function defined')

In [ ]:
# Cell 7: Pre-compute ERS for all claims (CPU-only, fast ~2-3 min)
print('Computing ERS for all 450 claims ...')

scores_all, indices_all = index.search(claim_embeddings.astype(np.float32), TOP_K_RETRIEVE)
all_ers_results = []

for i, claim in enumerate(claims):
    candidate_idxs   = indices_all[i].tolist()
    candidate_scores = scores_all[i].tolist()
    claim_emb = claim_embeddings[i]

    ers_ranked = compute_ers(
        claim_emb, candidate_idxs, candidate_scores,
        corpus_embeddings, evidence_corpus
    )
    all_ers_results.append(ers_ranked)

print(f'ERS computed for {len(all_ers_results)} claims')

# Sanity check — show top-3 ERS items for claim 0
print('\nTop-3 ERS for claim 0:')
for item in all_ers_results[0][:3]:
    idx = item['corpus_idx']
    ev  = evidence_corpus[idx]
    print(f'  ERS={item["ers"]:.4f}  rel={item["relevance"]:.4f}  '
          f'con={item["consistency"]:.4f}  type={ev["type"]}')
    print(f'    "{ev["text"][:120]}"')

In [ ]:
# Cell 8: Load Mistral-7B in 4-bit
MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'

print(f'Loading {MODEL_ID} in 4-bit ...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
model.eval()
print('Model loaded')

In [ ]:
# Cell 9: Prompt builders and generation helper

def extract_label(text):
    text_upper = text.upper()
    if 'NOT ENOUGH INFO' in text_upper or 'NOT ENOUGH INFORMATION' in text_upper or 'NEI' in text_upper:
        return 'NOT ENOUGH INFO'
    if 'REFUTES' in text_upper or 'REFUTED' in text_upper or 'FALSE' in text_upper:
        return 'REFUTES'
    if 'SUPPORTS' in text_upper or 'SUPPORTED' in text_upper or 'TRUE' in text_upper:
        return 'SUPPORTS'
    return 'UNKNOWN'


def build_proposed_prompt(claim_text, ers_evidence_list):
    """Prompt with evidence annotated by ERS reliability score and type."""
    ev_lines = []
    for j, item in enumerate(ers_evidence_list[:TOP_K_PROMPT]):
        ev     = evidence_corpus[item['corpus_idx']]
        ev_type = ev['type']
        ers_val = item['ers']
        ev_lines.append(
            f'[Evidence {j+1} | Type: {ev_type} | Reliability: {ers_val:.3f}] {ev["text"]}'
        )
    ev_block = '\n'.join(ev_lines)
    return (
        '[INST] You are a fact-checking assistant. Using ONLY the evidence below '
        '(higher Reliability score = more trustworthy), classify the claim as '
        'SUPPORTS, REFUTES, or NOT ENOUGH INFO. Prioritize higher-reliability evidence. '
        'Justify in 1-2 sentences.\n\n'
        f'Claim: {claim_text}\n\n'
        f'Evidence:\n{ev_block}\n\n'
        'Respond with:\nVerdict: <SUPPORTS|REFUTES|NOT ENOUGH INFO>\nJustification: <one or two sentences> [/INST]'
    )


def generate_response(prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(
        output_ids[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()


print('Prompt helpers defined')

In [ ]:
# Cell 10: Run Proposed pipeline
print('=== Running Proposed System (ERS-scored evidence + Mistral-7B) ===')
proposed_results = []

for i, claim in enumerate(tqdm(claims, desc='Proposed')):
    ers_ranked = all_ers_results[i]
    prompt     = build_proposed_prompt(claim['claim'], ers_ranked)

    t0 = time.time()
    response   = generate_response(prompt)
    latency    = round(time.time() - t0, 2)
    pred_label = extract_label(response)

    top_k_items = ers_ranked[:TOP_K_PROMPT]

    proposed_results.append({
        'claim_id':         claim['claim_id'],
        'claim':            claim['claim'],
        'gold_label':       claim['label'],
        'pred_label':       pred_label,
        'correct':          int(pred_label == claim['label']),
        'response':         response,
        'ers_ranked_top10': json.dumps(top_k_items),
        'mean_ers_top10':   round(float(np.mean([x['ers'] for x in top_k_items])), 5) if top_k_items else 0.0,
        'n_evidence_used':  len(top_k_items),
        'latency_s':        latency
    })

    if (i + 1) % 50 == 0:
        acc = sum(r['correct'] for r in proposed_results) / len(proposed_results)
        unk = sum(1 for r in proposed_results if r['pred_label'] == 'UNKNOWN') / len(proposed_results)
        print(f'  [{i+1}/{len(claims)}] acc={acc:.3f}  unknown_rate={unk:.3f}')

print(f'Proposed complete. Accuracy: {sum(r["correct"] for r in proposed_results)/len(proposed_results):.4f}')

In [ ]:
# Cell 11: Save proposed_outputs.csv
proposed_path = os.path.join(OUT_DIR, 'proposed_outputs.csv')
with open(proposed_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(proposed_results[0].keys()))
    writer.writeheader()
    writer.writerows(proposed_results)
print(f'Saved: {proposed_path}')

In [ ]:
# Cell 12: 3-way accuracy comparison
import pandas as pd

b1_df = pd.read_csv(os.path.join(OUT_DIR, 'b1_outputs.csv'))
b2_df = pd.read_csv(os.path.join(OUT_DIR, 'b2_outputs.csv'))
pr_df = pd.read_csv(proposed_path)

print('=' * 65)
print('LABEL ACCURACY COMPARISON  (B1 vs B2 vs Proposed)')
print('=' * 65)
systems = [
    ('B1 (LLM-only)',      b1_df),
    ('B2 (Standard RAG)',  b2_df),
    ('Proposed (ERS)',     pr_df)
]
for name, df in systems:
    acc = df['correct'].mean()
    unk = (df['pred_label'] == 'UNKNOWN').mean()
    print(f'\n{name}:  Accuracy = {acc:.4f}  Unknown rate = {unk:.4f}')
    for label in sorted(df['gold_label'].unique()):
        sub = df[df['gold_label'] == label]
        print(f'  {label:<24} {sub["correct"].mean():.4f}')
print('=' * 65)

In [ ]:
# Cell 13: Save ERS metadata for Day 5 (verification needs ERS-ranked items per claim)
ers_payload = [
    {'claim_id': claim['claim_id'], 'ers_ranked': all_ers_results[i]}
    for i, claim in enumerate(claims)
]

ers_metadata_path = os.path.join(DATA_DIR, 'ers_all_claims.json')
with open(ers_metadata_path, 'w', encoding='utf-8') as f:
    json.dump(ers_payload, f, indent=2)

print(f'ERS metadata saved: {ers_metadata_path}')

## Day 4 Complete

Files saved:
- `outputs/proposed_outputs.csv` — Proposed system results
- `data/ers_all_claims.json` — ERS-ranked evidence per claim (required by Day 5)

**ERS Implementation Note (paper Section 3.6):**
- `relevance_i = cosine_sim(claim_emb, evidence_emb)` normalized to [0,1]
- `consistency_i = mean cosine_sim(evidence_i, all_evidence_of_different_type)` normalized to [0,1]
- Three evidence types handled: `sentence`, `cell`, `table_row`
- `ERS_i = 0.7 * relevance_i + 0.3 * consistency_i`

Next: Open `Day5_Verification.ipynb`